# Lesson 03: Flow Matching for Tokens

In this notebook we adapt flow matching to generate **token sequences**. The pipeline:

1. Embed discrete tokens into continuous vectors.
2. Train flow matching on these embeddings.
3. Sample via ODE integration (Euler method).
4. Round back to discrete tokens via nearest-neighbor lookup.

This is the same idea as Diffusion-LM, but with an ODE instead of an SDE -- giving deterministic, faster generation.

In [ ]:
# Install dependencies
%pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import sys
sys.path.insert(0, "src")
from text_flow_matching import TextFlowMatcher

## 1. Setup: Synthetic Token Data

We create a small vocabulary and structured token sequences to demonstrate the pipeline. In practice, you would use a real tokenizer (e.g., GPT-2's BPE tokenizer) and real text data.

In [ ]:
# Simple vocabulary for demonstration
vocab = [
    "<pad>", "the", "a", "cat", "dog", "bird", "sat", "ran", "flew",
    "on", "in", "over", "mat", "box", "tree", "quickly", "slowly",
    "big", "small", "red", "blue", ".", ",", "and", "then",
]
vocab_size = len(vocab)
seq_len = 8

# Create structured training data -- simple sentence patterns
# Pattern: [det] [adj] [noun] [verb] [prep] [det] [adj] [noun]
def make_sentence():
    det = np.random.choice([1, 2])      # the, a
    adj = np.random.choice([17, 18, 19, 20])  # big, small, red, blue
    noun = np.random.choice([3, 4, 5])   # cat, dog, bird
    verb = np.random.choice([6, 7, 8])   # sat, ran, flew
    prep = np.random.choice([9, 10, 11]) # on, in, over
    det2 = np.random.choice([1, 2])
    adj2 = np.random.choice([17, 18, 19, 20])
    noun2 = np.random.choice([12, 13, 14])  # mat, box, tree
    return [det, adj, noun, verb, prep, det2, adj2, noun2]

train_data = torch.tensor([make_sentence() for _ in range(500)], device=device)
print(f"Training data shape: {train_data.shape}")
print(f"Example: {[vocab[i] for i in train_data[0].tolist()]}")
print(f"Example: {[vocab[i] for i in train_data[1].tolist()]}")

## 2. Create and Train TextFlowMatcher

In [ ]:
# Create the model -- using small dimensions for fast training
tfm = TextFlowMatcher(
    vocab_size=vocab_size,
    embed_dim=64,
    d_model=128,
    n_heads=4,
    n_layers=2,
    d_ff=256,
    seq_len=seq_len,
    lr=3e-4,
)
tfm.to(device)

params = tfm.count_parameters()
print(f"Parameters: {params}")

In [ ]:
# Train
batch_size = 64
losses = []

for step in range(500):
    idx = torch.randint(0, len(train_data), (batch_size,))
    batch = train_data[idx]
    loss = tfm.train_step(batch)
    losses.append(loss)
    
    if (step + 1) % 100 == 0:
        print(f"Step {step+1}: loss = {loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Training step")
plt.ylabel("Flow Matching Loss")
plt.title("TextFlowMatcher Training")
plt.show()

## 3. Generate Token Sequences

Sample embeddings via ODE, then round to nearest tokens.

In [ ]:
# Generate sequences
generated_ids = tfm.generate(batch_size=8, n_steps=50)

print("Generated sequences:")
for i in range(8):
    words = [vocab[j] for j in generated_ids[i].tolist()]
    print(f"  {i}: {' '.join(words)}")

## 4. Visualize the Generation Trajectory

Watch how tokens evolve from random noise to structured sequences.

In [ ]:
final_tokens, trajectory = tfm.generate_with_trajectory(
    batch_size=1, n_steps=100, save_every=10
)

print("Token evolution during generation (sample 0):")
print(f"{'t':>5} | Tokens")
print("-" * 60)
for tokens_at_t, t_val in trajectory:
    words = [vocab[j] for j in tokens_at_t[0].tolist()]
    print(f"{t_val:5.2f} | {' '.join(words)}")

words = [vocab[j] for j in final_tokens[0].tolist()]
print(f"{'final':>5} | {' '.join(words)}")

## 5. Comparing ODE Steps: Speed vs Quality

In [ ]:
step_counts = [10, 25, 50, 100, 200]
n_samples = 32

print(f"{'Steps':>6} | {'Time (ms)':>10} | Sample output")
print("-" * 70)

for n_steps in step_counts:
    start = time.time()
    tokens = tfm.generate(batch_size=n_samples, n_steps=n_steps)
    elapsed = (time.time() - start) * 1000
    
    words = [vocab[j] for j in tokens[0].tolist()]
    print(f"{n_steps:>6} | {elapsed:>10.1f} | {' '.join(words)}")

## 6. Deterministic Generation

A key advantage of ODE-based generation: the same initial noise always produces the same output.

In [ ]:
# Generate twice from the same noise
torch.manual_seed(123)
tokens_1 = tfm.generate(batch_size=4, n_steps=50)

torch.manual_seed(123)
tokens_2 = tfm.generate(batch_size=4, n_steps=50)

print("Same seed, run 1:")
for i in range(4):
    print(f"  {[vocab[j] for j in tokens_1[i].tolist()]}")

print("\nSame seed, run 2:")
for i in range(4):
    print(f"  {[vocab[j] for j in tokens_2[i].tolist()]}")

print(f"\nIdentical? {(tokens_1 == tokens_2).all().item()}")

## Summary

1. **TextFlowMatcher** adapts flow matching to token sequences by working in embedding space.
2. The pipeline: embed tokens -> train flow matching -> sample via ODE -> round to tokens.
3. ODE sampling is **deterministic** (same noise = same output) and typically needs **fewer steps** than SDE.
4. The rounding step (nearest-neighbor in embedding space) is shared with Diffusion-LM.

### Next: Lab

In the lab, you will build a complete flow-matching text generator on TinyStories and compare it against SDE-based generation.